# EDA & Baselines
Understand the reward metric, data distribution, and oracle upper bounds.


In [ ]:
# === ENVIRONMENT & FILEPATH SETUP ===
import os
import sys

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)

# Clear src modules to allow reloading
for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

DATA_DIR = f"{codebase_path}data"
CACHE_DIR = f"{codebase_path}output/cache"
MODELS_DIR = f"{codebase_path}output/models"
ARTIFACTS_DIR = f"{codebase_path}artifacts"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Load datasets
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()


In [ ]:
# Define models and custom reward metric calculation
models = ['Model_A', 'Model_B', 'Model_C', 'Model_D', 'Model_E', 'Model_F', 'Model_G', 'Model_H', 'Model_I', 'Model_J', 'Model_K']

def compute_reward(performance_list, cost_list, max_cost_list):
    avg_p = np.mean(performance_list)
    avg_c = np.mean(cost_list)
    avg_c_max = np.mean(max_cost_list)
    return 0.85 * avg_p - 0.15 * (avg_c / avg_c_max)



In [ ]:
# Extract performance and cost columns
perf_cols = [f"{m}_performance" for m in models]
cost_cols = [f"{m}_cost" for m in models]

max_cost_per_query = train[cost_cols].max(axis=1)
global_avg_max_cost = max_cost_per_query.mean()
print("Global Average Max Cost:", global_avg_max_cost)



In [ ]:
# Calculate Oracle Upper Bound
oracle_rewards = []
for i, row in train.iterrows():
    row_rewards = []
    for m in models:
        p = row[f"{m}_performance"]
        c = row[f"{m}_cost"]
        r = 0.85 * p - 0.15 * (c / global_avg_max_cost)
        row_rewards.append(r)
    oracle_rewards.append(max(row_rewards))

print("Oracle Upper Bound Average Reward:", np.mean(oracle_rewards))



In [ ]:
# Simple Baseline 1: Static Router
static_rewards = []
for m in models:
    avg_p = train[f"{m}_performance"].mean()
    avg_c = train[f"{m}_cost"].mean()
    r = 0.85 * avg_p - 0.15 * (avg_c / global_avg_max_cost)
    static_rewards.append((m, r))

static_rewards.sort(key=lambda x: x[1], reverse=True)
print("Static Router performance by model:")
for m, r in static_rewards:
    print(f"{m}: {r:.4f}")
print("\nBest Static Router:", static_rewards[0])

